# Phase 07F: Quantized Quality Aggregate and Deployment Lock

Phase 07F tổng hợp ba quality gate độc lập của T1–S1, T2–S2 và T3–S3,
kiểm tra lại toàn bộ bằng chứng, khóa lựa chọn INT8 và tạo bundle truy vết
cho bước đóng gói runtime Android hiện đại.

Phase này **không**:

- huấn luyện hoặc tinh chỉnh mô hình;
- lựa chọn lại checkpoint, seed hay ngưỡng quality gate;
- sử dụng Held-out Test;
- coi TorchScript/`torch.ao` hiện tại là gói Android cuối cùng;
- công bố benchmark CPU Kaggle là benchmark thiết bị vật lý.

Điều kiện đầu vào:

1. Phase 07D Mobile Runtime Quantization Preflight;
2. ba output Phase 07E đã hoàn tất;
3. Phase 01 Formal Output;
4. Phase 02A Shared Student Pipeline.

Runtime: CPU, Internet Off. Không cần MS COCO images hoặc GPU.


## Cell 1 — Authorization, manifests và ba quality gates


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//"
    "phase7f_quantized_quality_aggregate_deployment_lock",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
BUNDLE = OUT / "deployment_bundle"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, BUNDLE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)


def now():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents, (root, target)
    return target


def verify_csv_manifest(root, reference, expected_rows=None):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file()
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    if expected_rows is not None:
        assert len(rows) == int(expected_rows)
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows


PAIR_SPECS = [
    {
        "pair": "T1-S1", "slug": "t1_s1", "student_id": "S1", "seed": 123,
        "complete": "PHASE7E_T1_S1_QUANTIZED_VALIDATION_QUALITY_COMPLETE.json",
        "ready": "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_INPUT_READY_T1_S1.json",
    },
    {
        "pair": "T2-S2", "slug": "t2_s2", "student_id": "S2", "seed": 2026,
        "complete": "PHASE7E_T2_S2_QUANTIZED_VALIDATION_QUALITY_COMPLETE.json",
        "ready": "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_INPUT_READY_T2_S2.json",
    },
    {
        "pair": "T3-S3", "slug": "t3_s3", "student_id": "S3", "seed": 42,
        "complete": "PHASE7E_T3_S3_QUANTIZED_VALIDATION_QUALITY_COMPLETE.json",
        "ready": "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_INPUT_READY_T3_S3.json",
    },
]

P7D_PATH = exactly_one(
    "PHASE7D_MOBILE_RUNTIME_QUANTIZATION_PREFLIGHT_COMPLETE.json"
)
P7D_AUTH_PATH = exactly_one(
    "PHASE7E_VALIDATION_QUALITY_OR_FP32_PACKAGING_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P7D = read_json(P7D_PATH)
P7D_AUTH = read_json(P7D_AUTH_PATH)
P1 = read_json(P1_PATH)
P2 = read_json(P2_PATH)

assert P7D["status"] == "complete"
assert P7D["decision"] == (
    "compatibility_preflight_complete_phase7e_quantized_quality_authorized"
)
assert P7D_AUTH["status"] == "authorized"
assert P7D_AUTH["mode"] == "quantized_validation_quality_gate"
assert P7D_AUTH["new_training"] is False
assert P7D_AUTH["checkpoint_reselection"] is False
assert P7D_AUTH["test_set_status"] == "sealed"
assert P7D_AUTH["test_set_used"] is False
assert P1["status"] == "ready"
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2["status"] == "ready"
assert P2["data_policy"]["test_set_status"] == "sealed"
assert P2["data_policy"]["test_set_used"] is False

p7d_rows = verify_csv_manifest(
    P7D_PATH.parent, P7D["manifest"]["csv"], P7D["manifest_rows"]
)
EVIDENCE = {}
for spec in PAIR_SPECS:
    complete_path = exactly_one(spec["complete"])
    ready_path = exactly_one(spec["ready"])
    marker = read_json(complete_path)
    ready = read_json(ready_path)
    assert marker["status"] == "complete"
    assert marker["pair"] == spec["pair"]
    assert marker["student_id"] == spec["student_id"]
    assert int(marker["seed"]) == spec["seed"]
    assert marker["completed_images"] == marker["validation_images"] == 5000
    assert marker["quality_gate_passed"] is True
    assert marker["deployment_decision"] == "retain_int8_candidate_for_phase7f"
    assert marker["new_training_performed"] is False
    assert marker["checkpoint_reselection_performed"] is False
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
    rows = verify_csv_manifest(
        complete_path.parent,
        marker["manifest"]["csv"],
        marker["manifest_rows"],
    )
    assert ready["status"] == "ready"
    assert ready["pair"] == spec["pair"]
    assert ready["quality_gate_passed"] is True
    assert ready["deployment_decision"] == marker["deployment_decision"]
    assert ready["test_set_status"] == "sealed"
    assert ready["test_set_used"] is False
    quality_path = resolve_inside(
        ready_path.parent, ready["quality_result"]["relative_path"]
    )
    assert sha256(quality_path) == ready["quality_result"]["sha256"]
    quality = read_json(quality_path)
    assert quality["pair"] == spec["pair"]
    assert int(quality["seed"]) == spec["seed"]
    assert quality["validation_images"] == 5000
    assert quality["validation_references"] == 25010
    assert quality["quality_gate_passed"] is True
    assert quality["test_set_status"] == "sealed"
    assert quality["test_set_used"] is False
    EVIDENCE[spec["pair"]] = {
        "spec": spec,
        "complete_path": complete_path,
        "ready_path": ready_path,
        "marker": marker,
        "ready": ready,
        "quality_path": quality_path,
        "quality": quality,
        "manifest_rows": rows,
    }

write_json(AUDIT / "phase7f_cell01_entry.json", {
    "schema_version": "phase7f_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pairs": [spec["pair"] for spec in PAIR_SPECS],
    "phase7d_manifest": f"{len(p7d_rows)} / {P7D['manifest_rows']} verified",
    "phase7e_manifests": {
        pair: f"{len(item['manifest_rows'])} / "
        f"{item['marker']['manifest_rows']} verified"
        for pair, item in EVIDENCE.items()
    },
    "validation_images_per_pair": 5000,
    "validation_references_per_pair": 25010,
    "new_training_started": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "runtime": {
        "python": platform.python_version(),
        "cpu_only": True,
    },
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07F CELL 1 - ENTRY")
print("Status                  : PASSED")
print("Phase 07D manifest      :", len(p7d_rows), "/", P7D["manifest_rows"], "VERIFIED")
print("Phase 07E complete      : 3 / 3 VERIFIED")
print("Quality gates           : 3 / 3 PASSED")
print("Validation evidence     : 15,000 image evaluations")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 1 / 5 code cells")


PHASE 07F CELL 1 - ENTRY
Status                  : PASSED
Phase 07D manifest      : 23 / 23 VERIFIED
Phase 07E complete      : 3 / 3 VERIFIED
Quality gates           : 3 / 3 PASSED
Validation evidence     : 15,000 image evaluations
New training            : NOT STARTED
Held-out Test           : SEALED / NOT USED
Progress                : 1 / 5 code cells


## Cell 2 — Đối soát metric, quality gates và trade-off dung lượng


In [2]:
STARTED = time.perf_counter()
METRICS = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D", "CE"]
QUALITY_GATES = P7D_AUTH["quality_protocol"]["quantized_minus_fp32_gates"]
assert QUALITY_GATES == {
    "CIDEr-D_minimum_delta": -0.02,
    "BLEU-4_minimum_delta": -0.01,
    "CE_maximum_delta": 0.05,
}
selected_by_pair = {
    row["pair"]: row
    for row in P7D_AUTH["selected_quantized_candidates"]
}
assert set(selected_by_pair) == {spec["pair"] for spec in PAIR_SPECS}

SUMMARY_ROWS = []
for spec in PAIR_SPECS:
    pair = spec["pair"]
    quality = EVIDENCE[pair]["quality"]
    selected = selected_by_pair[pair]
    assert selected["student_id"] == spec["student_id"]
    assert int(selected["seed"]) == spec["seed"]
    assert selected["eligible_for_phase7e"] is True
    assert quality["strategy"] == selected["strategy"]
    assert all(
        metric in quality["fp32"]
        and metric in quality["int8"]
        and metric in quality["delta_int8_minus_fp32"]
        for metric in METRICS
    )
    for metric in METRICS:
        expected_delta = quality["int8"][metric] - quality["fp32"][metric]
        assert math.isclose(
            quality["delta_int8_minus_fp32"][metric],
            expected_delta,
            rel_tol=0.0,
            abs_tol=1e-12,
        )
        assert all(math.isfinite(value) for value in [
            quality["fp32"][metric],
            quality["int8"][metric],
            quality["delta_int8_minus_fp32"][metric],
        ])
    recomputed_gates = {
        "CIDEr-D": (
            quality["delta_int8_minus_fp32"]["CIDEr-D"]
            >= QUALITY_GATES["CIDEr-D_minimum_delta"]
        ),
        "BLEU-4": (
            quality["delta_int8_minus_fp32"]["BLEU-4"]
            >= QUALITY_GATES["BLEU-4_minimum_delta"]
        ),
        "CE": (
            quality["delta_int8_minus_fp32"]["CE"]
            <= QUALITY_GATES["CE_maximum_delta"]
        ),
        "finite": True,
    }
    assert recomputed_gates == quality["gates"]
    assert all(recomputed_gates.values())
    size_reduction = (
        1.0
        - selected["candidate_package_size_bytes"]
        / selected["fp32_package_size_bytes"]
    )
    assert math.isclose(
        size_reduction,
        selected["package_reduction_fraction"],
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    SUMMARY_ROWS.append({
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "strategy": selected["strategy"],
        "fp32_size_bytes": selected["fp32_package_size_bytes"],
        "int8_size_bytes": selected["candidate_package_size_bytes"],
        "size_reduction_fraction": size_reduction,
        "fp32_bleu4": quality["fp32"]["BLEU-4"],
        "int8_bleu4": quality["int8"]["BLEU-4"],
        "delta_bleu4": quality["delta_int8_minus_fp32"]["BLEU-4"],
        "fp32_cider_d": quality["fp32"]["CIDEr-D"],
        "int8_cider_d": quality["int8"]["CIDEr-D"],
        "delta_cider_d": quality["delta_int8_minus_fp32"]["CIDEr-D"],
        "fp32_ce": quality["fp32"]["CE"],
        "int8_ce": quality["int8"]["CE"],
        "delta_ce": quality["delta_int8_minus_fp32"]["CE"],
        "exact_caption_agreement_fraction": (
            quality["exact_caption_agreement_fraction"]
        ),
        "quality_gate_passed": True,
        "deployment_decision": "retain_int8",
    })

summary_csv = RESULT / "phase7f_quality_size_tradeoff.csv"
with summary_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(SUMMARY_ROWS[0]))
    writer.writeheader()
    writer.writerows(SUMMARY_ROWS)

aggregate = {
    "schema_version": "phase7f_quality_size_aggregate_v1",
    "created_at_utc": now(),
    "status": "passed",
    "selection_split": "Karpathy Validation",
    "pairs": SUMMARY_ROWS,
    "totals": {
        "pairs": len(SUMMARY_ROWS),
        "validation_images_evaluated": 5000 * len(SUMMARY_ROWS),
        "validation_references_used": 25010 * len(SUMMARY_ROWS),
        "fp32_size_bytes": sum(row["fp32_size_bytes"] for row in SUMMARY_ROWS),
        "int8_size_bytes": sum(row["int8_size_bytes"] for row in SUMMARY_ROWS),
    },
    "quality_gates": QUALITY_GATES,
    "test_set_status": "sealed",
    "test_set_used": False,
}
aggregate["totals"]["size_reduction_fraction"] = (
    1.0
    - aggregate["totals"]["int8_size_bytes"]
    / aggregate["totals"]["fp32_size_bytes"]
)
write_json(RESULT / "phase7f_quality_size_aggregate.json", aggregate)
write_json(AUDIT / "phase7f_cell02_metric_recomputation.json", {
    "schema_version": "phase7f_cell02_metric_recomputation_v1",
    "created_at_utc": now(),
    "status": "passed",
    "delta_recomputed": "18 / 18",
    "quality_gates_recomputed": "12 / 12",
    "package_reductions_recomputed": "3 / 3",
    "quality_passed": "3 / 3",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07F CELL 2 - QUALITY AND SIZE")
print("Status                  : PASSED")
for row in SUMMARY_ROWS:
    print(
        f"{row['pair']}: dCIDEr-D={row['delta_cider_d']:+.6f} | "
        f"dBLEU-4={row['delta_bleu4']:+.6f} | "
        f"dCE={row['delta_ce']:+.6f} | "
        f"size={row['size_reduction_fraction']:.2%}"
    )
print(
    "Aggregate size         :",
    f"{aggregate['totals']['size_reduction_fraction']:.2%} reduction",
)
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 2 / 5 code cells")


PHASE 07F CELL 2 - QUALITY AND SIZE
Status                  : PASSED
T1-S1: dCIDEr-D=-0.001906 | dBLEU-4=-0.001045 | dCE=+0.000058 | size=13.12%
T2-S2: dCIDEr-D=-0.004851 | dBLEU-4=-0.001943 | dCE=+0.000514 | size=29.75%
T3-S3: dCIDEr-D=+0.000383 | dBLEU-4=-0.000037 | dCE=+0.000064 | size=9.98%
Aggregate size         : 17.72% reduction
Held-out Test           : SEALED / NOT USED
Progress                : 2 / 5 code cells


## Cell 3 — Đóng gói artifact đã khóa và hợp đồng tiền xử lý


In [3]:
STARTED = time.perf_counter()
P7D_ROOT = P7D_PATH.parent
P1_ROOT = P1_PATH.parent
P2_ROOT = P2_PATH.parent

p2_rows = verify_csv_manifest(
    P2_ROOT, P2["manifests"]["csv"], P2["artifact_count"]
)
p2_by_relative_path = {row["relative_path"]: row for row in p2_rows}

phase1_sha_path = P1_ROOT / "SHA256SUMS.txt"
assert phase1_sha_path.is_file()
assert sha256(phase1_sha_path) == P1["sha256sums_sha256"]
phase1_hashes = {}
for line in phase1_sha_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    digest, relative_path = line.split(maxsplit=1)
    phase1_hashes[relative_path.strip()] = digest


def copy_verified(source, destination, expected_hash):
    source = Path(source)
    destination = Path(destination)
    assert source.is_file()
    assert sha256(source) == expected_hash
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert destination.stat().st_size == source.stat().st_size
    assert sha256(destination) == expected_hash
    return {
        "relative_path": destination.relative_to(OUT).as_posix(),
        "size_bytes": destination.stat().st_size,
        "sha256": expected_hash,
    }


VOCAB_RELATIVE = "vocabulary/word_vocabulary_minfreq5.json"
VOCAB_SOURCE = resolve_inside(P1_ROOT, VOCAB_RELATIVE)
assert sha256(VOCAB_SOURCE) == phase1_hashes[VOCAB_RELATIVE]
VOCAB_ARTIFACT = copy_verified(
    VOCAB_SOURCE,
    BUNDLE / "common/word_vocabulary_minfreq5.json",
    phase1_hashes[VOCAB_RELATIVE],
)

DECODER_RELATIVE = "configs/decoder/shared_transformer_decoder_contract.json"
decoder_manifest = p2_by_relative_path[DECODER_RELATIVE]
DECODER_ARTIFACT = copy_verified(
    resolve_inside(P2_ROOT, DECODER_RELATIVE),
    BUNDLE / "common/shared_transformer_decoder_contract.json",
    decoder_manifest["sha256"],
)

DEPLOYMENT_ROWS = []
for spec in PAIR_SPECS:
    pair = spec["pair"]
    slug = spec["slug"]
    selected = selected_by_pair[pair]
    artifact_map = {item["role"]: item for item in selected["artifacts"]}
    packaged_artifacts = {}
    for role in ["encoder", "decoder"]:
        source_info = artifact_map[role]
        source = resolve_inside(P7D_ROOT, source_info["relative_path"])
        packaged_artifacts[role] = copy_verified(
            source,
            BUNDLE / slug / f"{role}_int8_compatibility.ts",
            source_info["sha256"],
        )

    preprocessing_relative = (
        f"configs/preprocessing/{spec['student_id'].lower()}"
        "_preprocessing_contract.json"
    )
    preprocessing_manifest = p2_by_relative_path[preprocessing_relative]
    preprocessing_artifact = copy_verified(
        resolve_inside(P2_ROOT, preprocessing_relative),
        BUNDLE / slug / "preprocessing_contract.json",
        preprocessing_manifest["sha256"],
    )
    quality_source = EVIDENCE[pair]["quality_path"]
    quality_artifact = copy_verified(
        quality_source,
        BUNDLE / slug / "phase7e_quality_result.json",
        sha256(quality_source),
    )
    quality_contract_source = (
        EVIDENCE[pair]["complete_path"].parent
        / "configs/phase7e_pair_quality_contract.json"
    )
    quality_contract_artifact = copy_verified(
        quality_contract_source,
        BUNDLE / slug / "phase7e_pair_quality_contract.json",
        sha256(quality_contract_source),
    )
    summary = next(row for row in SUMMARY_ROWS if row["pair"] == pair)
    pair_contract = {
        "schema_version": "phase7f_locked_pair_contract_v1",
        "created_at_utc": now(),
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "selection_status": "locked",
        "selected_precision": "dynamic_int8_compatibility_candidate",
        "strategy": selected["strategy"],
        "artifacts": packaged_artifacts,
        "preprocessing": preprocessing_artifact,
        "vocabulary": VOCAB_ARTIFACT,
        "decoder_contract": DECODER_ARTIFACT,
        "quality_evidence": quality_artifact,
        "quality_contract": quality_contract_artifact,
        "quality_summary": summary,
        "runtime_status": {
            "legacy_torch_ao_compatibility_evidence": True,
            "modern_android_package_created": False,
            "physical_android_benchmark_performed": False,
        },
        "new_training_performed": False,
        "checkpoint_reselection_performed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    pair_contract_path = BUNDLE / slug / "locked_pair_contract.json"
    write_json(pair_contract_path, pair_contract)
    DEPLOYMENT_ROWS.append({
        "pair": pair,
        "student_id": spec["student_id"],
        "seed": spec["seed"],
        "strategy": selected["strategy"],
        "selected_precision": "dynamic_int8_compatibility_candidate",
        "encoder": packaged_artifacts["encoder"],
        "decoder": packaged_artifacts["decoder"],
        "pair_contract": {
            "relative_path": pair_contract_path.relative_to(OUT).as_posix(),
            "size_bytes": pair_contract_path.stat().st_size,
            "sha256": sha256(pair_contract_path),
        },
        "quality_gate_passed": True,
        "modern_android_package_created": False,
    })

write_json(RESULT / "phase7f_locked_deployment_registry.json", {
    "schema_version": "phase7f_locked_deployment_registry_v1",
    "created_at_utc": now(),
    "status": "locked",
    "rows": DEPLOYMENT_ROWS,
    "common": {
        "vocabulary": VOCAB_ARTIFACT,
        "decoder_contract": DECODER_ARTIFACT,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7f_cell03_bundle_integrity.json", {
    "schema_version": "phase7f_cell03_bundle_integrity_v1",
    "created_at_utc": now(),
    "status": "passed",
    "phase2_manifest": f"{len(p2_rows)} / {P2['artifact_count']} verified",
    "quantized_artifacts": "6 / 6 verified and copied",
    "preprocessing_contracts": "3 / 3 verified and copied",
    "quality_evidence": "3 / 3 verified and copied",
    "common_contracts": "2 / 2 verified and copied",
    "modern_android_package_created": False,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07F CELL 3 - LOCKED BUNDLE")
print("Status                  : PASSED")
print("INT8 compatibility      : 6 / 6 ARTIFACTS VERIFIED")
print("Preprocessing           : 3 / 3 VERIFIED")
print("Vocabulary / decoder    : 2 / 2 VERIFIED")
print("Modern Android package  : NOT CREATED")
print("Physical device test    : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 3 / 5 code cells")


PHASE 07F CELL 3 - LOCKED BUNDLE
Status                  : PASSED
INT8 compatibility      : 6 / 6 ARTIFACTS VERIFIED
Preprocessing           : 3 / 3 VERIFIED
Vocabulary / decoder    : 2 / 2 VERIFIED
Modern Android package  : NOT CREATED
Physical device test    : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Progress                : 3 / 5 code cells


## Cell 4 — Khóa lựa chọn và cấp quyền Phase 07G


In [4]:
STARTED = time.perf_counter()
assert len(DEPLOYMENT_ROWS) == 3
assert all(row["quality_gate_passed"] for row in DEPLOYMENT_ROWS)
assert all(
    row["selected_precision"] == "dynamic_int8_compatibility_candidate"
    for row in DEPLOYMENT_ROWS
)
runtime_policy = P7D_AUTH["runtime_policy"]
assert runtime_policy == {
    "legacy_torch_ao_result_is_compatibility_evidence_only": True,
    "modern_android_runtime_export_still_required": True,
    "physical_android_benchmark_required": True,
}

selection_lock = {
    "schema_version": "phase7f_deployment_selection_lock_v1",
    "created_at_utc": now(),
    "status": "locked",
    "decision": "retain_all_three_int8_compatibility_candidates",
    "pairs": [
        {
            "pair": row["pair"],
            "student_id": row["student_id"],
            "seed": row["seed"],
            "strategy": row["strategy"],
            "quality_gate_passed": row["quality_gate_passed"],
            "pair_contract": row["pair_contract"],
        }
        for row in DEPLOYMENT_ROWS
    ],
    "selection_split": "Karpathy Validation",
    "selection_change_allowed": False,
    "modern_android_runtime_export_required": True,
    "physical_android_benchmark_required": True,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(CONFIG / "phase7f_deployment_selection_lock.json", selection_lock)

phase7g_authorization = {
    "schema_version": "phase7g_modern_android_runtime_preflight_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "mode": "modern_android_runtime_packaging_compatibility_preflight",
    "required_next": (
        "Reconstruct each locked Student from the immutable source checkpoint; "
        "test a supported modern Android runtime export/quantization route; "
        "verify functional parity against the locked FP32/INT8 evidence; "
        "do not benchmark a physical device until runtime packages pass."
    ),
    "locked_pairs": [row["pair"] for row in DEPLOYMENT_ROWS],
    "locked_seeds": {
        row["pair"]: row["seed"] for row in DEPLOYMENT_ROWS
    },
    "compatibility_artifacts_are_final_android_packages": False,
    "modern_android_package_required": True,
    "functional_parity_required": True,
    "physical_android_benchmark_authorized": False,
    "physical_android_benchmark_required_after_packaging": True,
    "new_training": False,
    "checkpoint_reselection": False,
    "quality_gate_change_allowed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    OUT / "PHASE7G_MODERN_ANDROID_RUNTIME_PACKAGING_PREFLIGHT_AUTHORIZED.json",
    phase7g_authorization,
)
write_json(AUDIT / "phase7f_cell04_decision.json", {
    "schema_version": "phase7f_cell04_decision_v1",
    "created_at_utc": now(),
    "status": "passed",
    "quality_gates": "3 / 3 passed",
    "deployment_selection": "locked",
    "phase7g": "authorized",
    "physical_android_benchmark": "not_yet_authorized",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07F CELL 4 - DECISION")
print("Status                  : PASSED")
print("Deployment selection    : LOCKED")
print("INT8 candidates         : 3 / 3 RETAINED")
print("Phase 07G               : AUTHORIZED")
print("Physical device test    : NOT YET AUTHORIZED")
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 4 / 5 code cells")


PHASE 07F CELL 4 - DECISION
Status                  : PASSED
Deployment selection    : LOCKED
INT8 candidates         : 3 / 3 RETAINED
Phase 07G               : AUTHORIZED
Physical device test    : NOT YET AUTHORIZED
New training            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Progress                : 4 / 5 code cells


## Cell 5 — Manifest và completion marker


In [5]:
STARTED = time.perf_counter()
manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, BUNDLE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })

manifest_csv = MANIFEST / "phase7f_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7f_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7f_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)

for row in manifest_rows:
    path = resolve_inside(OUT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256(path) == row["sha256"]

completion = {
    "schema_version": (
        "phase7f_quantized_quality_aggregate_deployment_lock_complete_v1"
    ),
    "created_at_utc": now(),
    "status": "complete",
    "decision": (
        "all_three_int8_quality_gates_passed_"
        "phase7g_modern_runtime_preflight_authorized"
    ),
    "quality_gates_passed": "3 / 3",
    "deployment_pairs_locked": ["T1-S1", "T2-S2", "T3-S3"],
    "validation_images_evaluated": 15000,
    "validation_references_used": 75030,
    "selected_precision": {
        row["pair"]: row["selected_precision"] for row in DEPLOYMENT_ROWS
    },
    "modern_android_package_created": False,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_csv.relative_to(OUT).as_posix(),
            "size_bytes": manifest_csv.stat().st_size,
            "sha256": sha256(manifest_csv),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": manifest_json.relative_to(OUT).as_posix(),
            "size_bytes": manifest_json.stat().st_size,
            "sha256": sha256(manifest_json),
        },
        "sha256sums": {
            "relative_path": sha_file.relative_to(OUT).as_posix(),
            "size_bytes": sha_file.stat().st_size,
            "sha256": sha256(sha_file),
        },
    },
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell05": time.perf_counter() - STARTED,
}
write_json(
    OUT / "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_"
    "DEPLOYMENT_LOCK_COMPLETE.json",
    completion,
)

print("PHASE 07F CELL 5 - COMPLETE")
print("Status                  : COMPLETE")
print("Quality gates           : 3 / 3 PASSED")
print("Deployment pairs        : 3 / 3 LOCKED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Modern Android package  : NOT YET CREATED")
print("Physical device test    : NOT YET PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Next                    : Phase 07G modern runtime packaging preflight")
print("Progress                : 5 / 5 code cells")


PHASE 07F CELL 5 - COMPLETE
Status                  : COMPLETE
Quality gates           : 3 / 3 PASSED
Deployment pairs        : 3 / 3 LOCKED
Manifest                : 28 / 28 VERIFIED
Modern Android package  : NOT YET CREATED
Physical device test    : NOT YET PERFORMED
Held-out Test           : SEALED / NOT USED
Next                    : Phase 07G modern runtime packaging preflight
Progress                : 5 / 5 code cells
